In [1]:
import keras
from keras import models
from keras import layers
from keras.datasets import mnist
from keras.utils import to_categorical
import numpy as np
import tensorflow as tf

2026-03-06 11:49:48.693759: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-06 11:49:48.693960: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 11:49:48.721388: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 11:49:49.319741: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [2]:
(train_images, train_labels), (test_images, test_labels)= mnist.load_data()
#Print dataset dimensions
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [3]:
#Add number of channels
train_images=train_images.reshape(train_images.shape[0], 28, 28, 1)
test_images=test_images.reshape(test_images.shape[0], 28, 28, 1)

#Normalize images
train_images=train_images.astype('float32')/255.0
test_images=test_images.astype('float32')/255.0

train_labels=to_categorical(train_labels, 10)
test_labels=to_categorical(test_labels, 10)

In [4]:
#First parameter in Conv2D = number of filters = number of features

#Problem: creates a lot of weights
# model = models.Sequential([
#     layers.Conv2D(14, kernel_size=(3, 3), input_shape=(28, 28, 1), strides=(1,1), padding="valid", use_bias = True), 
#     layers.Activation("relu"),
#     layers.MaxPooling2D(pool_size=(2, 2)),
#     layers.Conv2D(32, kernel_size=(3, 3), strides=(1,1), padding="valid", use_bias = True),
#     layers.Activation("relu"),
#     layers.MaxPooling2D(pool_size=(2, 2)),
#     layers.Flatten(),
#     layers.Dropout(0.5), 
#     layers.Dense(10, activation="softmax"),
# ])

In [ ]:
#During the 28x28 test, there was a problem that the accumulator would saturate too quickly (hence why you see the debug prints in the Ada library and requantization registers exposed in VHDL)
#Google search (AI search at this point tbh) suggested using fake quantization and max value parameters, so I tried it
#Among so many things, I think this technique did help (or maybe it was the fixing of the export script in convert_weightsv4.py). I kept it because it makes sense intuitively. By limiting the intermitten values, the model is trained on Q0.7 numbers
#Another technique suggested and incorporated was regularizers. It keeps model weights small, which is necessary to not saturate the accumulator

#Decorator to serialize the quantization function in the saved model file
#Not doing so caused erros when opening the model in the convert_weights script
@keras.saving.register_keras_serializable()
def fake_q07(x):
    x_clip = tf.clip_by_value(x, -1.0, 127.0/128.0)
    x_q = tf.round(x_clip * 128.0) / 128.0
    return x_clip + tf.stop_gradient(x_q - x_clip)

model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(14, (3, 3), padding="valid", use_bias=True,
                  kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    layers.ReLU(max_value=127.0/128.0),
    layers.Lambda(fake_q07),

    layers.MaxPooling2D((2, 2)),
    layers.Lambda(fake_q07),

    layers.Conv2D(32, (3, 3), padding="valid", use_bias=True,
                  kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    layers.ReLU(max_value=127.0/128.0),
    layers.Lambda(fake_q07),

    layers.MaxPooling2D((2, 2)),
    layers.Lambda(fake_q07),

    layers.Flatten(),
    layers.Dropout(0.2),

    layers.Dense(10, activation="softmax",
                 kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
])
model.summary()

E0000 00:00:1772777990.242924    8817 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1772777990.249334    8817 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 14)     │           140 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 26, 26, 14)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 26, 26, 14)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 14)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 13, 13, 14)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 32)     │         4,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_3 (Lambda)               │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         8,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,214 (47.71 KB)

 Trainable params: 12,214 (47.71 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
#Lighter model
#Output shape of a convolutional layer: [(W−K+2P)/S]+1.
#https://stackoverflow.com/questions/53580088/calculate-the-output-size-in-convolution-layer
#W is the input image width
#K is the Kernel size
#P is the padding = 0
#S is the stride = 1

#For max pooling layer output: https://keras.io/2/api/layers/pooling_layers/max_pooling2d/
#output_shape = math.floor((input_shape - pool_size) / strides) + 1 (when input_shape >= pool_size)
#Stride defaults to pool size
# model = models.Sequential([
#     layers.Conv2D(4, kernel_size=(3, 3), input_shape=(28, 28, 1), strides=(1,1), padding="valid", use_bias = True),
#     #output shape = [(28-3)/1+1] = 26x26x4 = 2704 (4 comes from number of filters)
    
#     layers.Activation("relu"),
#     layers.MaxPooling2D(pool_size=(2, 2)),
#     #Output shape = [(26-2)/2] + 1 = 13x13x8 = 1352
    
#     layers.Conv2D(8, kernel_size=(3, 3), strides=(1,1), padding="valid", use_bias = True),
#     #output shape = [(13-3)/1+1] = 11x11x8 = 968 (8 comes from number of filters)
    
#     layers.Activation("relu"),
#     layers.MaxPooling2D(pool_size=(2, 2)),
#     #Output shape = [(11-2)/2] + 1 = 6x6x8 = 288 (or 5x5x8 = 200)
    
#     layers.Flatten(),  #6x6x8 = 288 (or 5x5x8 = 200)
#     layers.Dropout(0.5),
#     layers.Dense(10), #288*10 = 2880 or 200*10 = 2000 
#     layers.Activation("softmax")
#     #Total weights/biases = 2704 + 1352 + 968 + 288 + 288 + 2880 = 8.4KB
# ])

In [ ]:
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", 
        patience=5,              
        min_delta=0.001,          
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", 
        factor=0.5, 
        patience=2               
    ),
]


model.fit(train_images, train_labels, epochs=100,batch_size=128,verbose=2,validation_split=0.3, callbacks=callbacks)

Epoch 1/100
329/329 - 4s - 12ms/step - accuracy: 0.8564 - loss: 0.5156 - val_accuracy: 0.9473 - val_loss: 0.1959 - learning_rate: 0.0010
Epoch 2/100
329/329 - 4s - 11ms/step - accuracy: 0.9556 - loss: 0.1636 - val_accuracy: 0.9653 - val_loss: 0.1284 - learning_rate: 0.0010
Epoch 3/100
329/329 - 3s - 10ms/step - accuracy: 0.9666 - loss: 0.1224 - val_accuracy: 0.9709 - val_loss: 0.1125 - learning_rate: 0.0010
Epoch 4/100
329/329 - 3s - 10ms/step - accuracy: 0.9721 - loss: 0.1061 - val_accuracy: 0.9742 - val_loss: 0.0985 - learning_rate: 0.0010
Epoch 5/100
329/329 - 3s - 10ms/step - accuracy: 0.9759 - loss: 0.0960 - val_accuracy: 0.9757 - val_loss: 0.0945 - learning_rate: 0.0010
Epoch 6/100
329/329 - 3s - 10ms/step - accuracy: 0.9783 - loss: 0.0894 - val_accuracy: 0.9807 - val_loss: 0.0842 - learning_rate: 0.0010
Epoch 7/100
329/329 - 3s - 10ms/step - accuracy: 0.9799 - loss: 0.0846 - val_accuracy: 0.9802 - val_loss: 0.0833 - learning_rate: 0.0010
Epoch 8/100
329/329 - 3s - 10ms/step - ac

In [8]:
#Evaluate model
test_loss, test_acc = model.evaluate(test_images, test_labels)
print("Test accuracy: ", test_acc)

print(model.get_weights())
model.save(filepath="28x28_mnist_larger_model.keras")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9855 - loss: 0.0619
Test accuracy:  0.9854999780654907
[array([[[[-0.02609275,  0.3615702 ,  0.10033549, -0.00147778,
           0.03602583,  0.03745703,  0.22002676, -0.61207074,
          -0.5265241 ,  0.12250843, -0.49413496, -0.11124128,
           0.13053517,  0.42371938]],

        [[ 0.26529804,  0.05330285,  0.05341374,  0.24609172,
          -0.0673388 ,  0.36429375, -0.3323471 , -0.16715874,
          -0.17115329,  0.16856591, -0.29289782, -0.33287463,
           0.16872637,  0.34579322]],

        [[ 0.2833818 , -0.18983601,  0.1424773 , -0.0253463 ,
           0.06146964,  0.48166415, -0.33382353,  0.13655096,
           0.31093216, -0.03218213, -0.00212326, -0.5283014 ,
           0.43072584, -0.0570417 ]]],


       [[[-0.3299566 , -0.04409002,  0.20010817,  0.20871167,
           0.1382657 , -0.47526106,  0.16649616, -0.19291937,
          -0.44637665,  0.19662236,  0.096752  ,  0.23283556,
           0.31749544,  0.

In [9]:
def quantize_q07(x):
    x_clipped=np.clip(x, -1.0, 0.9921875) #Restrict weights in the numpy array to -1 to 127/128
    q=np.round(x_clipped * 128.0).astype(np.int16) #128=scaling factor
                                                     #int16 is used in the intermediate step 
                                                     #similar to our calculations in VHDL
    q=np.clip(q, -128, 127).astype(np.int8)       #Clip to int8
    return q

In [10]:
from pathlib import Path
import random
ada_out_dir = Path("exported_samples_q07")
ada_out_dir.mkdir(parents=True, exist_ok=True)
ada_ads_path = ada_out_dir / "mnist_test_samples_28x28.ads"
ada_package_name = "mnist_test_samples_28x28"

In [11]:
def write_ada_int_array(f, name, flat):
    f.write(f"  {name} : constant Integer_Array (Natural range 0 .. {len(flat)-1}) := (\n")
    for i, w in enumerate(flat):
        sep = "," if i != len(flat) - 1 else ""
        f.write(f"  {int(w)}{sep}\n")
    f.write("  );\n")

In [12]:
#Convert tensors to numpy arrays for indexing
#train_vec_np = train_vec.numpy()          #(N, 196)
test_vec  = tf.reshape(test_images,  [-1, 28 * 28])  
train_vec_np = test_vec.numpy()
train_labels_np = np.array(test_labels)  #N

sample_count = 10

random_numbers = [random.randint(0, len(train_vec_np)-1) for _ in range(sample_count)]

random_images=[]
random_labels=[]

for index in random_numbers:
    random_images.append(quantize_q07(train_vec_np[index])  )
    random_labels.append(int(np.argmax(train_labels_np[index])))
  

with open(ada_ads_path, "w") as f:
    f.write("with Input_Output_Helper; use Input_Output_Helper;\n")
    f.write(f"package {ada_package_name} is\n")
    f.write(f"  Sample_Count : constant Natural := {sample_count};\n")
    f.write(f"  type Sample_Set is array (Natural range 0 .. Sample_Count - 1) of Integer_Array (Natural range 0 .. 783);\n")

    #Write Samples as a proper 2D Ada constant
    f.write("   Samples : constant Sample_Set := (\n")
    for s in range(sample_count):
        f.write(f"  {s} => (\n")
        flat = random_images[s].astype(np.int8).flatten()  # ensure Python int printing is sane
        for i, v in enumerate(flat):
            if(i!=len(flat)-1):
                sep = ","
            else:
                sep = ""
            f.write(f"  {int(v)}{sep}")
            if((i + 1) % 20 == 0):
                f.write("\n")
            else:
                f.write(" ")
        closer="\n)"
        if(s != sample_count - 1):
            closer = closer + ","
        closer = closer + "\n"
        f.write(closer)
    f.write(");\n")

    write_ada_int_array(f,"    Labels", random_labels)

    f.write(f"end {ada_package_name};\n")

print(f"Wrote Ada samples to: {ada_ads_path}")


Wrote Ada samples to: exported_samples_q07/mnist_test_samples_28x28.ads
